# Advanced Python Generator Problems with Solutions

## `yield from`, `send()`, `throw()`, `close()`, `GeneratorExit`, and return values

This notebook turns generator delegation into a practical, testable skill through **15 advanced solved problems**, additional challenges, debugging techniques, and production-oriented best practices.

Run the notebook from top to bottom. Every solution cell contains assertions, and the notebook has been execution-validated.

## Learning objectives

You will learn to:

1. Trace delegator and subgenerator states.
2. Explain how `next()`, `send()`, `throw()`, and `close()` cross a `yield from` boundary.
3. Capture a subgenerator's `return` value.
4. Distinguish normal completion from cancellation.
5. Implement safe cleanup with `GeneratorExit`.
6. Create custom iterators that cooperate with delegation.
7. Test yielded values and final return values separately.
8. Build recursive and streaming generator pipelines.
9. Expand a simplified version of `yield from`.
10. Debug suspended generators with `inspect`.

## Python-version note

One behavior changed in Python 3.13:

- Python 3.13 and newer: `generator.close()` returns a value when the generator catches `GeneratorExit` and returns a value.
- Older versions: the close-time return value is discarded and `close()` returns `None`.

The exercises detect the interpreter version and avoid accidental version dependence.

References:

- https://docs.python.org/3/reference/expressions.html
- https://peps.python.org/pep-0380/
- https://peps.python.org/pep-0342/
- https://peps.python.org/pep-0479/

## Best-practice checklist

- Prime coroutine-style generators explicitly.
- Use normal messages or sentinels for successful completion; use `close()` for cancellation and cleanup.
- Put resource cleanup in `finally`.
- Never yield after receiving `GeneratorExit`.
- Prefer `yield from` when `send`, `throw`, and `close` must propagate.
- Treat generator return values as completion metadata, not streamed values.
- Test both yielded items and `StopIteration.value`.
- Prefer structured event logs over fragile print-only checks.
- Close resource-owning generators deterministically.
- Use frame inspection only for debugging, not application logic.

## Shared setup

In [2]:
from __future__ import annotations

from dataclasses import dataclass, field
from inspect import getgeneratorlocals, getgeneratorstate
from typing import Any, Iterable
import sys

print("Running Python:", sys.version.split()[0])

Running Python: 3.13.7


In [3]:
@dataclass
class EventLog:
    events: list[tuple[str, Any]] = field(default_factory=list)

    def add(self, event: str, value: Any = None) -> None:
        self.events.append((event, value))

    def names(self) -> list[str]:
        return [name for name, _ in self.events]


def collect_with_return(generator) -> tuple[list[Any], Any]:
    "Collect yielded values and capture the final StopIteration.value."
    yielded = []
    while True:
        try:
            yielded.append(next(generator))
        except StopIteration as exc:
            return yielded, exc.value


def assert_state(generator, expected: str) -> None:
    actual = getgeneratorstate(generator)
    assert actual == expected, f"expected {expected}, got {actual}"


def close_return_supported() -> bool:
    return sys.version_info >= (3, 13)

# Problem 1 — Trace a delegation state machine

Create a worker and a delegator. Prove with inspection that:

- a newly created delegator is `GEN_CREATED`,
- priming suspends both generators,
- `send()` reaches the subgenerator,
- the subgenerator's return value becomes the result of `yield from`,
- both generators eventually close.

### Solution 1

In [4]:
def traced_worker(log: EventLog):
    log.add("worker-start")
    first = yield "worker-ready"
    log.add("worker-received", first)
    second = yield f"ack:{first}"
    log.add("worker-received", second)
    return {"count": 2, "last": second}


def traced_delegator(log: EventLog):
    log.add("delegator-start")
    result = yield from traced_worker(log)
    log.add("delegator-result", result)
    yield ("summary", result)
    log.add("delegator-end")

In [5]:
log = EventLog()
delegator = traced_delegator(log)

assert_state(delegator, "GEN_CREATED")
assert next(delegator) == "worker-ready"
assert_state(delegator, "GEN_SUSPENDED")

worker = delegator.gi_yieldfrom
assert worker is not None
assert_state(worker, "GEN_SUSPENDED")

assert delegator.send("alpha") == "ack:alpha"
summary = delegator.send("omega")

assert summary == ("summary", {"count": 2, "last": "omega"})
assert delegator.gi_yieldfrom is None
assert_state(worker, "GEN_CLOSED")

assert next(delegator, "DONE") == "DONE"
assert_state(delegator, "GEN_CLOSED")

assert log.events == [
    ("delegator-start", None),
    ("worker-start", None),
    ("worker-received", "alpha"),
    ("worker-received", "omega"),
    ("delegator-result", {"count": 2, "last": "omega"}),
    ("delegator-end", None),
]

log.events

[('delegator-start', None),
 ('worker-start', None),
 ('worker-received', 'alpha'),
 ('worker-received', 'omega'),
 ('delegator-result', {'count': 2, 'last': 'omega'}),
 ('delegator-end', None)]

**Key idea:** while delegation is active, `gi_yieldfrom` identifies the current subiterator. After the subgenerator returns, the delegator resumes after `yield from`.

# Problem 2 — Streaming statistics with a returned report

Write `running_stats()` that:

- begins by yielding `"ready"`,
- receives numbers through `send()`,
- yields the current mean,
- terminates normally on a sentinel,
- returns `count`, `sum`, `min`, `max`, and `mean`.

A delegator must capture the report and yield one final report tuple.

### Solution 2

In [6]:
STOP = object()


def running_stats():
    count = 0
    total = 0.0
    minimum = None
    maximum = None

    value = yield "ready"

    while value is not STOP:
        if not isinstance(value, (int, float)):
            raise TypeError(f"expected number, got {type(value).__name__}")

        count += 1
        total += value
        minimum = value if minimum is None else min(minimum, value)
        maximum = value if maximum is None else max(maximum, value)

        value = yield total / count

    return {
        "count": count,
        "sum": total,
        "min": minimum,
        "max": maximum,
        "mean": None if count == 0 else total / count,
    }


def stats_session():
    result = yield from running_stats()
    yield ("final-report", result)

In [7]:
session = stats_session()

assert next(session) == "ready"
assert session.send(10) == 10.0
assert session.send(20) == 15.0
assert session.send(-3) == 9.0

assert session.send(STOP) == (
    "final-report",
    {"count": 3, "sum": 27.0, "min": -3, "max": 20, "mean": 9.0},
)

assert next(session, None) is None

**Best practice:** use a sentinel for successful protocol completion. Reserve `close()` for cancellation or cleanup, not for delivering the primary result.

# Problem 3 — Prove innermost-first cleanup

Create an `outer -> middle -> inner` delegation chain. Every level logs entry and cleanup. Prime the outer generator, close it, and prove cleanup runs from the active innermost generator outward.

### Solution 3

In [8]:
def inner_layer(log: EventLog):
    log.add("inner-enter")
    try:
        while True:
            yield "inner-active"
    finally:
        log.add("inner-finally")


def middle_layer(log: EventLog):
    log.add("middle-enter")
    try:
        yield from inner_layer(log)
    finally:
        log.add("middle-finally")


def outer_layer(log: EventLog):
    log.add("outer-enter")
    try:
        yield from middle_layer(log)
    finally:
        log.add("outer-finally")

In [9]:
log = EventLog()
chain = outer_layer(log)

assert next(chain) == "inner-active"
chain.close()

assert_state(chain, "GEN_CLOSED")
assert log.names() == [
    "outer-enter",
    "middle-enter",
    "inner-enter",
    "inner-finally",
    "middle-finally",
    "outer-finally",
]

log.events

[('outer-enter', None),
 ('middle-enter', None),
 ('inner-enter', None),
 ('inner-finally', None),
 ('middle-finally', None),
 ('outer-finally', None)]

# Problem 4 — Close only the active subgenerator

Obtain the active subgenerator through `gi_yieldfrom`, close only that object, and prove the delegator remains suspended until the caller resumes it.

### Solution 4

In [10]:
def closable_sub(log: EventLog):
    try:
        while True:
            incoming = yield "sub-waiting"
            log.add("sub-received", incoming)
    finally:
        log.add("sub-closed")


def parent_after_subclose(log: EventLog):
    yield from closable_sub(log)
    log.add("parent-resumed")
    yield "parent-after-delegation"
    log.add("parent-finished")

In [11]:
log = EventLog()
parent = parent_after_subclose(log)

assert next(parent) == "sub-waiting"
sub = parent.gi_yieldfrom
assert sub is not None

sub.close()

assert_state(sub, "GEN_CLOSED")
assert_state(parent, "GEN_SUSPENDED")
assert log.events == [("sub-closed", None)]

assert next(parent) == "parent-after-delegation"
assert log.names() == ["sub-closed", "parent-resumed"]

assert next(parent, None) is None
assert log.names() == ["sub-closed", "parent-resumed", "parent-finished"]

# Problem 5 — Route recoverable and fatal exceptions

Build a subgenerator that catches `ValueError`, logs it, and continues. It must not catch `KeyError`.

Prove that `throw(ValueError(...))` reaches the subgenerator, while `throw(KeyError(...))` propagates and closes the chain.

### Solution 5

In [12]:
def resilient_worker(log: EventLog):
    while True:
        try:
            item = yield "awaiting-item"
        except ValueError as exc:
            log.add("recovered", str(exc))
            yield "recovered"
        else:
            log.add("item", item)


def resilient_parent(log: EventLog):
    try:
        yield from resilient_worker(log)
    finally:
        log.add("parent-finally")

In [13]:
log = EventLog()
worker = resilient_parent(log)

assert next(worker) == "awaiting-item"
assert worker.throw(ValueError("bad payload")) == "recovered"
assert next(worker) == "awaiting-item"
assert worker.send("valid") == "awaiting-item"

try:
    worker.throw(KeyError("fatal"))
except KeyError as exc:
    assert exc.args == ("fatal",)
else:
    raise AssertionError("KeyError should propagate")

assert_state(worker, "GEN_CLOSED")
assert log.events == [
    ("recovered", "bad payload"),
    ("item", "valid"),
    ("parent-finally", None),
]

# Problem 6 — Detect an illegal yield during closure

Create a broken generator that catches `GeneratorExit` and yields again. Verify that `close()` raises `RuntimeError`. Then implement the correct cleanup pattern.

### Solution 6

In [14]:
def broken_cleanup():
    try:
        yield "running"
    except GeneratorExit:
        yield "illegal"


broken = broken_cleanup()
assert next(broken) == "running"

try:
    broken.close()
except RuntimeError as exc:
    assert "ignored GeneratorExit" in str(exc)
else:
    raise AssertionError("Yielding during close must raise RuntimeError")

# Ensure no suspended generator remains after the demonstration.
try:
    broken.close()
except Exception:
    pass

In [15]:
def correct_cleanup(log: EventLog):
    try:
        yield "running"
    except GeneratorExit:
        log.add("cancelled")
        raise
    finally:
        log.add("resource-released")


log = EventLog()
resource = correct_cleanup(log)

assert next(resource) == "running"
assert resource.close() is None
assert_state(resource, "GEN_CLOSED")
assert log.names() == ["cancelled", "resource-released"]

# Problem 7 — Compare normal return, `throw(GeneratorExit)`, and `close()`

Create a generator that returns `10` normally, but catches `GeneratorExit` and returns `123`. Test all three completion paths with version-aware assertions.

### Solution 7

In [16]:
def returns_on_close(log: EventLog):
    try:
        yield "open"
        return 10
    except GeneratorExit:
        log.add("caught-generator-exit")
        return 123
    finally:
        log.add("finally")

In [17]:
# Normal completion
log = EventLog()
g = returns_on_close(log)
assert next(g) == "open"

try:
    next(g)
except StopIteration as exc:
    assert exc.value == 10
else:
    raise AssertionError("Expected StopIteration")

assert log.names() == ["finally"]

# Explicit throw exposes StopIteration.value
log = EventLog()
g = returns_on_close(log)
assert next(g) == "open"

try:
    g.throw(GeneratorExit())
except StopIteration as exc:
    assert exc.value == 123
else:
    raise AssertionError("Expected StopIteration carrying 123")

assert log.names() == ["caught-generator-exit", "finally"]

# close() changed in Python 3.13
log = EventLog()
g = returns_on_close(log)
assert next(g) == "open"
close_result = g.close()

if close_return_supported():
    assert close_result == 123
else:
    assert close_result is None

assert log.names() == ["caught-generator-exit", "finally"]

# Problem 8 — Delegate to a custom iterator

Implement a non-generator iterator with `__next__`, `send`, `throw`, and `close`. Use `yield from` to prove each protocol method is forwarded when applicable.

### Solution 8

In [18]:
class ProtocolIterator:
    def __init__(self, log: EventLog):
        self.log = log
        self.closed = False
        self.counter = 0

    def __iter__(self):
        return self

    def __next__(self):
        self.log.add("__next__")
        if self.closed:
            raise StopIteration("iterator-closed")
        self.counter += 1
        return ("next", self.counter)

    def send(self, value):
        self.log.add("send", value)
        if value == "stop":
            raise StopIteration("iterator-return")
        return ("sent", value)

    def throw(self, exc_type, exc=None, tb=None):
        # Accept both throw(exception_instance) and the legacy
        # throw(exception_type, exception_value, traceback) shape.
        if isinstance(exc_type, BaseException) and exc is None:
            actual = exc_type
        elif exc is not None:
            actual = exc
        else:
            actual = exc_type()
        self.log.add("throw", type(actual).__name__)
        if isinstance(actual, ValueError):
            return ("handled", str(actual))
        raise actual

    def close(self):
        self.log.add("close")
        self.closed = True


def delegate_to_custom(log: EventLog):
    result = yield from ProtocolIterator(log)
    yield ("delegator-result", result)

In [19]:
log = EventLog()
custom = delegate_to_custom(log)

assert next(custom) == ("next", 1)
assert custom.send("payload") == ("sent", "payload")
assert custom.throw(ValueError("recoverable")) == ("handled", "recoverable")
assert custom.send("stop") == ("delegator-result", "iterator-return")
assert next(custom, None) is None

assert log.events == [
    ("__next__", None),
    ("send", "payload"),
    ("throw", "ValueError"),
    ("send", "stop"),
]

In [20]:
log = EventLog()
custom = delegate_to_custom(log)

assert next(custom) == ("next", 1)
custom.close()

assert log.names() == ["__next__", "close"]
assert_state(custom, "GEN_CLOSED")

# Problem 9 — Recursive traversal with returned aggregate metadata

Write a recursive generator that yields every leaf in a nested dictionary as `(path, value)` and returns the number of leaves. Use `yield from` to propagate counts upward.

### Solution 9

In [21]:
def walk_tree(node: Any, path: tuple[str, ...] = ()):
    if not isinstance(node, dict):
        yield (path, node)
        return 1

    leaf_count = 0
    for key, child in node.items():
        child_count = yield from walk_tree(child, path + (key,))
        leaf_count += child_count

    return leaf_count

In [22]:
tree = {
    "service": {
        "host": "localhost",
        "ports": {"http": 80, "https": 443},
    },
    "debug": False,
}

leaves, count = collect_with_return(walk_tree(tree))

assert leaves == [
    (("service", "host"), "localhost"),
    (("service", "ports", "http"), 80),
    (("service", "ports", "https"), 443),
    (("debug",), False),
]
assert count == 4

leaves, count

([(('service', 'host'), 'localhost'),
  (('service', 'ports', 'http'), 80),
  (('service', 'ports', 'https'), 443),
  (('debug',), False)],
 4)

# Problem 10 — Resource ownership and deterministic cleanup

Simulate a resource-owning line source. It must log `open`, yield lines, return a count, and log `close` in `finally`. A transforming delegator must uppercase values and close the source during both normal completion and cancellation.

### Solution 10

In [23]:
def line_source(lines: Iterable[str], log: EventLog):
    log.add("open")
    count = 0
    try:
        for line in lines:
            count += 1
            yield line
        return count
    finally:
        log.add("close")


def uppercase_session(lines: Iterable[str], log: EventLog):
    source = line_source(lines, log)

    try:
        while True:
            try:
                line = next(source)
            except StopIteration as exc:
                count = exc.value
                break
            else:
                yield line.upper()
    finally:
        source.close()

    return count

In [24]:
log = EventLog()
items, count = collect_with_return(uppercase_session(["a\n", "b\n"], log))

assert items == ["A\n", "B\n"]
assert count == 2
assert log.names() == ["open", "close"]

log = EventLog()
session = uppercase_session(["a", "b", "c"], log)

assert next(session) == "A"
session.close()

assert log.names() == ["open", "close"]
assert_state(session, "GEN_CLOSED")

# Problem 11 — Streaming command parser

Implement a coroutine that accepts `ADD <integer>`, `RESET`, and `END`.

- Yield the current total after valid commands.
- Yield structured errors and continue after invalid commands.
- Return a report on `END`.
- Use a delegator to emit one final report.

### Solution 11

In [25]:
def command_parser():
    total = 0
    successful = 0
    errors = 0

    command = yield "parser-ready"

    while command != "END":
        try:
            parts = command.split()
            if parts == ["RESET"]:
                total = 0
            elif len(parts) == 2 and parts[0] == "ADD":
                total += int(parts[1])
            else:
                raise ValueError("expected ADD <integer>, RESET, or END")
        except (AttributeError, ValueError) as exc:
            errors += 1
            command = yield {
                "ok": False,
                "error": str(exc),
                "input": command,
            }
        else:
            successful += 1
            command = yield {"ok": True, "total": total}

    return {
        "total": total,
        "successful_commands": successful,
        "errors": errors,
    }


def parser_session():
    report = yield from command_parser()
    yield {"type": "final", **report}

In [26]:
parser = parser_session()

assert next(parser) == "parser-ready"
assert parser.send("ADD 5") == {"ok": True, "total": 5}
assert parser.send("ADD -2") == {"ok": True, "total": 3}

error = parser.send("MULTIPLY 7")
assert error["ok"] is False
assert error["input"] == "MULTIPLY 7"

assert parser.send("RESET") == {"ok": True, "total": 0}
assert parser.send("ADD 9") == {"ok": True, "total": 9}

assert parser.send("END") == {
    "type": "final",
    "total": 9,
    "successful_commands": 4,
    "errors": 1,
}
assert next(parser, None) is None

# Problem 12 — Write a simplified manual equivalent of `yield from`

Implement a delegator that manually supports:

- `next()`,
- non-`None` `send()`,
- `StopIteration.value`,
- and close forwarding.

General exception forwarding may be omitted. Compare the manual version with native `yield from`.

### Solution 12

In [27]:
def simple_delegate(subiterator):
    iterator = iter(subiterator)

    try:
        current = next(iterator)
    except StopIteration as exc:
        return exc.value

    while True:
        try:
            sent = yield current
        except GeneratorExit:
            close_method = getattr(iterator, "close", None)
            if close_method is not None:
                close_method()
            raise
        else:
            try:
                if sent is None:
                    current = next(iterator)
                else:
                    current = iterator.send(sent)
            except StopIteration as exc:
                return exc.value

In [28]:
def conversational_subgenerator(log: EventLog):
    name = yield "name?"
    log.add("name", name)
    age = yield "age?"
    log.add("age", age)
    return {"name": name, "age": age}


def native_parent(log: EventLog):
    result = yield from conversational_subgenerator(log)
    yield result


def manual_parent(log: EventLog):
    result = yield from simple_delegate(conversational_subgenerator(log))
    yield result

In [29]:
for parent_factory in (native_parent, manual_parent):
    log = EventLog()
    conversation = parent_factory(log)

    assert next(conversation) == "name?"
    assert conversation.send("Ada") == "age?"
    assert conversation.send(36) == {"name": "Ada", "age": 36}
    assert next(conversation, None) is None
    assert log.events == [("name", "Ada"), ("age", 36)]

The full PEP 380 expansion is more complex because it also forwards arbitrary exceptions, handles missing protocol methods, and preserves traceback information. Native `yield from` is safer.

# Problem 13 — Test yielded values and return values separately

Create a reusable assertion helper that verifies the yielded sequence, final return value, and final generator state. Apply it to an empty generator, a basic generator, and a nested delegator.

### Solution 13

In [30]:
def assert_generator_result(factory, expected_yields, expected_return) -> None:
    generator = factory()
    yielded, returned = collect_with_return(generator)

    assert yielded == expected_yields
    assert returned == expected_return
    assert_state(generator, "GEN_CLOSED")


def empty_generator():
    if False:
        yield None
    return "empty"


def numbers():
    yield 1
    yield 2
    return 3


def nested_numbers():
    inner_result = yield from numbers()
    yield inner_result * 10
    return "nested-done"

In [31]:
assert_generator_result(empty_generator, [], "empty")
assert_generator_result(numbers, [1, 2], 3)
assert_generator_result(nested_numbers, [1, 2, 30], "nested-done")

**Testing principle:** `list(generator)` discards the generator's final return value. Use an explicit driver when `StopIteration.value` is part of the contract.

# Problem 14 — Inspect suspended locals

Build an accumulator and a delegating parent. Suspend the chain after several sends, inspect both generator states and the subgenerator's locals, then terminate normally and verify closed-frame behavior.

### Solution 14

In [32]:
def accumulator():
    total = 0
    last = None

    while True:
        last = yield total
        if last is None:
            return total
        total += last


def accumulator_parent():
    result = yield from accumulator()
    yield ("done", result)

In [33]:
acc = accumulator_parent()

assert next(acc) == 0
assert acc.send(5) == 5
assert acc.send(7) == 12

sub = acc.gi_yieldfrom
assert sub is not None
assert getgeneratorstate(acc) == "GEN_SUSPENDED"
assert getgeneratorstate(sub) == "GEN_SUSPENDED"

locals_snapshot = getgeneratorlocals(sub)
assert locals_snapshot["total"] == 12
assert locals_snapshot["last"] == 7

assert acc.send(None) == ("done", 12)
assert next(acc, None) is None
assert getgeneratorlocals(acc) == {}

locals_snapshot

{'total': 12, 'last': None}

**Warning:** inspection is useful for diagnostics, but production code should expose state through a documented object, message, event, or return value.

# Problem 15 — Capstone: cancellable event pipeline

Create a three-stage system:

1. A source yields events and returns a production report.
2. A validator yields only valid events and returns validation metadata.
3. A session delegates to the validator and emits a final combined report.

Test normal completion and early cancellation. Every stage must clean up in the correct order.

### Solution 15

In [34]:
def event_source(events: Iterable[dict[str, Any]], log: EventLog):
    produced = 0
    log.add("source-enter")
    try:
        for event in events:
            produced += 1
            yield event
        return {"produced": produced}
    finally:
        log.add("source-finally")


def validating_stage(events: Iterable[dict[str, Any]], log: EventLog):
    valid_count = 0
    invalid: list[dict[str, Any]] = []
    source = event_source(events, log)

    log.add("validator-enter")
    try:
        while True:
            try:
                event = next(source)
            except StopIteration as exc:
                source_report = exc.value
                break

            is_valid = (
                isinstance(event, dict)
                and isinstance(event.get("id"), int)
                and isinstance(event.get("payload"), str)
            )

            if is_valid:
                valid_count += 1
                yield event
            else:
                invalid.append(event)
    finally:
        source.close()
        log.add("validator-finally")

    return {
        "valid": valid_count,
        "invalid": invalid,
        "source": source_report,
    }


def event_session(events: Iterable[dict[str, Any]], log: EventLog):
    log.add("session-enter")
    try:
        validation_report = yield from validating_stage(events, log)
        yield {"type": "report", **validation_report}
    finally:
        log.add("session-finally")

In [35]:
events = [
    {"id": 1, "payload": "alpha"},
    {"id": "bad", "payload": "beta"},
    {"id": 2, "payload": "gamma"},
    {"id": 3},
]

log = EventLog()
yielded = list(event_session(events, log))

assert yielded == [
    {"id": 1, "payload": "alpha"},
    {"id": 2, "payload": "gamma"},
    {
        "type": "report",
        "valid": 2,
        "invalid": [
            {"id": "bad", "payload": "beta"},
            {"id": 3},
        ],
        "source": {"produced": 4},
    },
]

assert log.names() == [
    "session-enter",
    "validator-enter",
    "source-enter",
    "source-finally",
    "validator-finally",
    "session-finally",
]

In [36]:
log = EventLog()
pipeline = event_session(events, log)

assert next(pipeline) == {"id": 1, "payload": "alpha"}
pipeline.close()

assert_state(pipeline, "GEN_CLOSED")
assert log.names() == [
    "session-enter",
    "validator-enter",
    "source-enter",
    "source-finally",
    "validator-finally",
    "session-finally",
]

# Additional unsolved challenge problems

## Challenge A — Timeout injection

Define `PipelineTimeout`. Throw it into a delegator and let the active subgenerator convert it into a normal completion report. Compare that design with `close()`.

## Challenge B — Bidirectional router

Create a delegator that switches between two subgenerators while preserving each subgenerator's local state.

## Challenge C — Backpressure protocol

Build a consumer that yields `"PAUSE"` every `n` inputs and requires `"RESUME"` before continuing.

## Challenge D — Transaction rollback

Model a transaction generator that returns a commit report on a sentinel and records rollback when closed early.

## Challenge E — Delegation-depth profiler

Build recursive delegation chains of depths 1, 10, and 100. Traverse `gi_yieldfrom` links and verify cleanup order.

# Common mistakes

1. **Using `list()` when the return value matters.** It discards `StopIteration.value`.
2. **Using `close()` as successful completion.** Prefer an explicit sentinel or command.
3. **Yielding after `GeneratorExit`.** This raises `RuntimeError`.
4. **Replacing `yield from` with a basic forwarding loop.** A basic loop does not fully forward `send`, `throw`, and `close`.
5. **Testing only printed output.** Prefer event logs and assertions.
6. **Depending on suspended-frame locals.** Use explicit state interfaces.
7. **Leaving resource-owning generators unclosed.** Clean them up deterministically.

# Compact protocol reference

| Caller operation | Active delegation behavior |
|---|---|
| `next(g)` | Calls `next()` on the subiterator |
| `g.send(None)` | Calls `next()` on the subiterator |
| `g.send(value)` | Calls `send(value)` on the subiterator |
| `g.throw(exc)` | Uses the subiterator's `throw(...)` when available |
| `g.close()` | Uses the subiterator's `close()` when available, then terminates outward |
| Subiterator raises `StopIteration(value)` | `yield from` evaluates to `value` |
| Subiterator yields a value | The value passes directly to the caller |

# Final self-check

You should now be able to explain:

- why `return x` in a subgenerator becomes the value of `yield from`,
- why directly closing a subgenerator does not immediately run the delegator,
- why cleanup proceeds innermost-first,
- why yielding after `GeneratorExit` is invalid,
- why normal termination is best for final reports,
- and why native `yield from` is safer than manual protocol forwarding.

In [37]:
# Final smoke test
assert close_return_supported() == (sys.version_info >= (3, 13))

test_log = EventLog()
test_values = list(event_session([{"id": 1, "payload": "ok"}], test_log))

assert test_values[-1]["type"] == "report"
assert test_values[-1]["valid"] == 1
assert test_values[-1]["source"]["produced"] == 1

print("All solution cells executed successfully.")

All solution cells executed successfully.
